# Workflow: Feature Selection

**What is the motivation?**

Many Reinforcement Learning environments have high-dimensional observation spaces. With increasing state space dimensionality, learning an effective policy typically requires more and more data and time.

**What do we show here?**

Using `VeriGym`, we can easily specify which state features we want to exclude from our enviroment and automatically construct a reduced environment, including an abstraction mapping between the original and reduced observation spaces.

We use the Mujoco `"Ant-v5"` robotics environment from [`gymnasium`](https://gymnasium.farama.org/environments/mujoco/ant/) as an example. Its observation space has 105 dimension. The first 27 correspond to the torso pose and joint positions and velocities. The latter 78 dimensions correspond to the center of mass based external forces on the pody parts.

In this workflow notebook, we reduce the observation space to the first 27 dimensions by masking the remaining features. We then train a SAC agent using `stable_baselines3` on both the full and the reduced observation spaces.
We compare:
- the training performance on the original environment versus the reduced environment
- the performance of the policy trained on the reduced environment when evaluated on the original environment.

## Imports and helpers

In [ ]:
# Imports
import gymnasium as gym
import stable_baselines3 as sb3

# VeriGym
import verigym
import verigym.abstraction.feature_selection as fs
from verigym.frameworks.stable_baselines3.policy import SB3Policy

from workflow_utils import plot_logged_data, run_eval_episodes, plot_compare_policy_eval#, train_with_sb3_sac

# Paths etc.
outpath = "../examples/workflow_fr_files/"

## Actual workflow

First, we load the standard `"Ant-v5"` environment into `VeriGym` using `verigym.GenerativeEnv.from_gymnasium`. 
This maintains all functionality of the original `gym` environment, and adds compatibility with additional `VeriGym` features.

In [ ]:
ant_env = verigym.GenerativeEnv.from_gymnasium(
    gym.make("Ant-v5", render_mode="rgb_array")
)

Then, we specify the features to reduce (here: the 78 features that are not related to position or velocity). 
Following the description of `Ant`'s observation space, the first 13 elements refer to the positions of the robot's body parts, and the next 14 elements to the velocities of these body parts.
The latter 78 express external forces.
Intuitively, if the robot should only learn to walk, could it be enough to use first 27 then?

We try this using `VeriGym`'s feature selection with masking to remove the selected features.

In [ ]:
reduce_features = [i for i in range(27, 105)]
fr_masked_env, abstraction_map = fs.state_feature_selection(original_env=ant_env, method="masking",
                                 reduce_indices=reduce_features)

For reference, we compare the original and reduced environments' action and observation spaces, as well as types.

Notice that the observation space dimensionality changes. Both environments' types are `VeriGym` native and `gym` compatible. Action spaces remain the same.

In [ ]:
print("Original environment:")
print("type:", type(ant_env))
print("observation space: ", ant_env.observation_space)
print("action space: ", ant_env.action_space)
print()
print("Reduced environment:")
print("type:", type(fr_masked_env))
print("observation space: ", fr_masked_env.observation_space)
print("action space: ", fr_masked_env.action_space)

## Training code

We train models using the `stable_baselines3` implementation of `SAC` for `1.000.000` timesteps.

Note: Training took 70-85min on a 2023 MacBook Pro with 18GB RAM and M3 Chip. Therefore, the training lines are commented and we proceed with pre-saved models and logs.
To run training yourself, comment the lines back in.

In [ ]:
#train_with_sb3_sac(ant_env, outpath+"full_env/")

In [ ]:
#train_with_sb3_sac(fr_masked_env, outpath+"reduced_env/")

## Evaluation

Using the pre-trained models, we compare the following aspects: 
1. Training behavior (reward and episode length) between the agents trained on the environment with 105 dimensions (`full_model`) versus the reduced one (`reduced_model`).
2. Policy transfer by mapping the policy trained on the reduced environment back to the original observation space using `VeriGym`'s `SB3Policy` wrapper and the automatically generated `AbstractionMapper`.


In [ ]:
full_model = sb3.SAC.load(outpath + "full_env/sac_model")
reduced_model = sb3.SAC.load(outpath + "reduced_env/sac_model")

In [ ]:
plot_logged_data([outpath + "full_env/log/evaluations.npz", outpath + "reduced_env/log/evaluations.npz"],
                    labels=["full env", "reduced env"], key="results",
                    title="Training results using SAC", eval_step=10000)

plot_logged_data([outpath + "full_env/log/evaluations.npz", outpath + "reduced_env/log/evaluations.npz"],
                    labels=["full env", "reduced env"], key="ep_lengths",
                    title="Training ep_lengths using SAC",
                    eval_step=10000)

Now, we compare policy performance by wrapping the stable baselines policies in to a `verigym.SB3Policy`. `full_policy` wraps the policy learned on the full observation space. `reduced_policy` wraps the policy learned on the reduced observation space.

Additionally, we can use the `abstraction_map` we got when we applied the feature reduction to have `VeriGym` map the policy from the reduced feature space to the original full feature space - we do that in `reduced_on_full_policy`.

In [ ]:
full_policy = SB3Policy(full_model)
reduced_policy = SB3Policy(reduced_model)
reduced_on_full_policy = SB3Policy(reduced_model, abstraction_map)

We now evaluate the full policy and the reduced policy on the original full environment.

In [ ]:
full_eval = run_eval_episodes(ant_env, full_policy)
reduced_on_full_eval = run_eval_episodes(ant_env, reduced_on_full_policy)
reduced_eval = run_eval_episodes(fr_masked_env, reduced_policy)

In [ ]:
plot_compare_policy_eval([full_eval, reduced_on_full_eval, reduced_eval], ["full policy & env", "reduced policy on full env", "reduced policy & env "])